# 05 — Compustat Fundamentals Panel

**Goal:** Construct a firm-year panel of accounting fundamentals for the S&P 500
universe. Provides the dependent variables for the operating-performance test (H2b):
ROA, operating margin, and sales growth.

**Input:**
- `data/sp500_universe_with_gvkey.parquet` (712 unique gvkeys)
- `comp.funda` — Compustat North America annual fundamentals

**Output:** `data/fundamentals_panel.parquet`
- One row per (gvkey, fyear)
- Operating-performance outcomes: ROA, operating margin, sales growth
- Raw inputs: net income, total assets, revenue, COGS, operating income before D&A
- Controls: log assets, leverage; book-to-market deferred to merged panel (needs ret

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import wrds

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"

universe = pd.read_parquet(DATA_PROCESSED / "sp500_universe_with_gvkey.parquet")
target_gvkeys = sorted(universe["gvkey"].dropna().unique())
print(f"Target gvkeys: {len(target_gvkeys)}")

Target gvkeys: 717


In [2]:
# Read the WRDS username from ~/.pgpass (4th field) so the notebook runs headlessly
WRDS_USERNAME = next(
    line.split(":")[3]
    for line in (Path.home() / ".pgpass").read_text().splitlines()
    if "wrds" in line
)
db = wrds.Connection(wrds_username=WRDS_USERNAME)

Loading library list...


Done


In [3]:
gvkey_list_sql = ",".join([f"'{g}'" for g in target_gvkeys])

funda_query = f"""
    SELECT  gvkey,
            datadate,
            fyear,
            indfmt,
            datafmt,
            popsrc,
            consol,
            sich,
            ni,
            at,
            revt,
            cogs,
            oibdp,
            lt,
            seq
    FROM    comp.funda
    WHERE   gvkey IN ({gvkey_list_sql})
      AND   fyear BETWEEN 2011 AND 2024
      AND   indfmt  = 'INDL'
      AND   datafmt = 'STD'
      AND   popsrc  = 'D'
      AND   consol  = 'C'
"""

print("Running query — usually under a minute...")
funda = db.raw_sql(funda_query)
print(f"Pulled {len(funda):,} firm-year rows")
print(f"Unique gvkeys: {funda['gvkey'].nunique()}")
print()

# Verify no duplicates (the four filters should prevent any)
dup_check = funda.duplicated(subset=["gvkey", "fyear"]).sum()
print(f"Duplicate (gvkey, fyear) rows: {dup_check}")

Running query — usually under a minute...


Pulled 9,048 firm-year rows
Unique gvkeys: 717

Duplicate (gvkey, fyear) rows: 0


In [4]:
# Coerce date
funda["datadate"] = pd.to_datetime(funda["datadate"])

# Operating-performance outcomes
funda["roa"] = funda["ni"] / funda["at"]
funda["operating_margin"] = funda["oibdp"] / funda["revt"]
# Gross margin — only valid for non-financial firms
funda["gross_margin"] = (funda["revt"] - funda["cogs"]) / funda["revt"]

# Sales growth — needs lagged revenue within each firm.
# FIX 2026-07-12: replaced groupby().pct_change() with an explicit fiscal-year
# merge. pct_change's default pad-fill forward-filled missing revenues
# (fabricating growth values around NaN-revenue firm-years), and its positional
# differencing would silently span fiscal-year gaps. Growth is now strictly
# revt(fyear) / revt(fyear - 1) - 1, NaN whenever either year is missing.
# The stored fundamentals_panel.parquet was patched in place with this same
# logic (src/patch_sales_growth_20260712.py) rather than re-executing the WRDS
# pull, to keep the Compustat vintage frozen; stored outputs below predate the fix.
funda = funda.sort_values(["gvkey", "fyear"]).reset_index(drop=True)
_prev = funda[["gvkey", "fyear", "revt"]].copy()
_prev["fyear"] = _prev["fyear"] + 1
funda = funda.merge(_prev.rename(columns={"revt": "revt_prev"}),
                    on=["gvkey", "fyear"], how="left", validate="m:1")
funda["sales_growth"] = funda["revt"] / funda["revt_prev"] - 1
funda = funda.drop(columns=["revt_prev"])

# Controls
funda["log_at"] = np.log(funda["at"])
funda["leverage"] = funda["lt"] / funda["at"]

# Quick sanity check — distributions
print("Outcome variable distributions:")
print(funda[["roa", "operating_margin", "gross_margin", "sales_growth"]].describe().round(3))
print()

print("Control variable distributions:")
print(funda[["log_at", "leverage"]].describe().round(3))
print()

# How many financial firms (SIC 6000-6999)?
funda["sich"] = pd.to_numeric(funda["sich"], errors="coerce")
financials = funda[(funda["sich"] >= 6000) & (funda["sich"] < 7000)]
print(f"Financial-firm observations (SIC 6000-6999): {len(financials):,} "
      f"({len(financials)/len(funda):.1%})")
print(f"Unique financial gvkeys: {financials['gvkey'].nunique()}")
print()

# Verify that financials have missing gross margin
print(f"Gross margin missing rate among financials: {financials['gross_margin'].isna().mean():.1%}")
print(f"Gross margin missing rate among non-financials: "
      f"{funda[~funda.index.isin(financials.index)]['gross_margin'].isna().mean():.1%}")

Outcome variable distributions:
          roa  operating_margin  gross_margin  sales_growth
count  9014.0            8590.0        9014.0        8306.0
mean    0.056             0.233         0.436         0.083
std     0.097             0.232         0.274         0.376
min    -2.283            -8.549        -8.549        -0.904
25%     0.019              0.14         0.277        -0.011
50%     0.051              0.22         0.413         0.051
75%     0.094             0.329         0.599         0.129
max     1.496             0.952         1.108        21.991

Control variable distributions:
       log_at  leverage
count  9016.0    9003.0
mean    9.728     0.648
std     1.445     0.259
min     3.894     0.032
25%     8.783     0.492
50%     9.647     0.637
75%     10.61     0.785
max    15.203      4.35

Financial-firm observations (SIC 6000-6999): 1,690 (18.7%)
Unique financial gvkeys: 133

Gross margin missing rate among financials: 0.1%
Gross margin missing rate among non-fina

/var/folders/2j/hjkjj6xd441624gv0_y769_w0000gn/T/ipykernel_62863/3015105323.py:12: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  funda["sales_growth"] = funda.groupby("gvkey")["revt"].pct_change()


In [5]:
# What does gross margin look like for financials vs. non-financials?
financials_mask = (funda["sich"] >= 6000) & (funda["sich"] < 7000)

print("Gross margin — FINANCIALS:")
print(funda[financials_mask]["gross_margin"].describe().round(3))
print()
print("Gross margin — NON-FINANCIALS:")
print(funda[~financials_mask]["gross_margin"].describe().round(3))
print()

# Look at sample financial firm-years to see what cogs looks like
print("Sample financial firm-years (gvkey, fyear, revt, cogs, gross_margin):")
print(funda[financials_mask][["gvkey", "fyear", "revt", "cogs", "gross_margin"]].head(15))

Gross margin — FINANCIALS:
count    1689.0
mean      0.428
std       0.272
min      -1.306
25%       0.215
50%        0.37
75%       0.617
max       1.108
Name: gross_margin, dtype: Float64

Gross margin — NON-FINANCIALS:
count    7295.0
mean      0.438
std       0.275
min      -8.549
25%       0.295
50%        0.42
75%       0.598
max         1.0
Name: gross_margin, dtype: Float64

Sample financial firm-years (gvkey, fyear, revt, cogs, gross_margin):
      gvkey  fyear     revt     cogs  gross_margin
56   001177   2011  33779.8  23530.0       0.30343
57   001177   2012  36595.9  26678.4         0.271
58   001177   2013  47284.9  35191.4      0.255758
59   001177   2014  58003.2  42911.7      0.260184
60   001177   2015  60226.9  43832.6      0.272209
61   001177   2016  63155.0  46356.0      0.265996
62   001177   2017  60447.0  44628.0        0.2617
150  001447   2011  32282.0  23770.0      0.263676
151  001447   2012  33808.0  24423.0      0.277597
152  001447   2013  34932.0  25051

In [6]:
# Keep the columns needed downstream
final = funda[[
    "gvkey", "fyear", "datadate", "sich",
    "ni", "at", "revt", "cogs", "oibdp", "lt", "seq",
    "roa", "operating_margin", "gross_margin", "sales_growth",
    "log_at", "leverage",
]].sort_values(["gvkey", "fyear"]).reset_index(drop=True)

print(f"Final panel: {len(final):,} rows")
print(f"Unique gvkeys: {final['gvkey'].nunique()}")
print(f"Year range: {final['fyear'].min()} to {final['fyear'].max()}")
print()

# Save
output_path = DATA_PROCESSED / "fundamentals_panel.parquet"
final.to_parquet(output_path, index=False)

# Verify
check = pd.read_parquet(output_path)
print(f"Saved {len(check):,} rows to {output_path.name}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print()

# Coverage check against universe
universe = pd.read_parquet(DATA_PROCESSED / "sp500_universe_with_gvkey.parquet")
universe["gvkey"] = universe["gvkey"].astype(str)
check["gvkey"] = check["gvkey"].astype(str)

universe_check = universe.merge(
    check[["gvkey", "fyear", "roa"]].rename(columns={"fyear": "year"}),
    on=["gvkey", "year"],
    how="left"
)
matched = universe_check["roa"].notna().sum()
print(f"Universe firm-year rows: {len(universe_check):,}")
print(f"  Matched in fundamentals: {matched:,} ({matched/len(universe_check):.1%})")
print()

# Coverage by year
print("Coverage by year:")
coverage = universe_check.groupby("year").agg(
    universe_firms=("gvkey", "nunique"),
    with_fundamentals=("roa", lambda x: x.notna().sum()),
).assign(pct=lambda d: d["with_fundamentals"] / d["universe_firms"])
print(coverage.round(3))

Final panel: 9,048 rows
Unique gvkeys: 717
Year range: 2011 to 2024

Saved 9,048 rows to fundamentals_panel.parquet
File size: 950.2 KB

Universe firm-year rows: 6,535
  Matched in fundamentals: 6,509 (99.6%)

Coverage by year:
      universe_firms  with_fundamentals    pct
year                                          
2012             497              494.0  0.994
2013             497              494.0  0.994
2014             496              496.0    1.0
2015             497              499.0  1.004
2016             500              502.0  1.004
2017             499              503.0  1.008
2018             500              503.0  1.006
2019             500              504.0  1.008
2020             500              503.0  1.006
2021             500              504.0  1.008
2022             500              503.0  1.006
2023             500              503.0  1.006
2024             500              501.0  1.002
